#### SEMANTiCS 2026 — demo compile

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str((Path.cwd() / ".." / "src").resolve()))

from rdflib import Graph
from rdfine import GraphDict
from compilers import (
    CompilationConfig,
    CompilationRunner,
    FileMaterializer,
    PipelineGeneratorConfig,
)

#### Load catalog

In [ ]:
catalog_file = Path("catalog_demo.ttl")
catalog = Graph().parse(catalog_file)  # for the inspection cell below

#### Compile & write

In [ ]:
config = CompilationConfig(
    compilers=PipelineGeneratorConfig.compilers,
    graph_files=[catalog_file],
    inference_files=PipelineGeneratorConfig.inference_files,
)
build = CompilationRunner("demo:SemanticsDemoPipeline", config).compile()

# materialises the pipeline build to disk
FileMaterializer(build).write("out")

#### Inspect pipeline definition in catalog

In [4]:
from rdfine import GraphReader

sub = (
    GraphReader(catalog)
    .traverse(
        "demo:SemanticsDemoPipeline",
        against="p-plan:isStepOfPlan",
        prune=["prov:specializationOf", "p-plan:hasInputVar", "tcs:writesTo", "tcs:readsFrom"],
    )
    .filter(pred=["rdf:type", "rdfs:label", "p-plan:isPrecededBy"])
)
print(GraphDict(sub.graph).serialize("yaml", "drop"))

'@graph':
- '@id': Poll
  '@type':
  - InstancePipelineComponent
- '@id': SemanticsDemoPipeline
  '@type':
  - PipelineDefinition
  label:
  - SEMANTiCS 2026 demo pipeline
- '@id': Log
  '@type':
  - InstancePipelineComponent
  isPrecededBy:
  - '@id': Parse
- '@id': Parse
  '@type':
  - InstancePipelineComponent
  isPrecededBy:
  - '@id': Poll

